![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Day 24 -- Lab 1: Road Signs with YOLO

An Olympiad-style detection task: 877 street photos from Kaggle, each with an XML file listing its signs (traffic light, stop, speed limit, crosswalk) and their boxes, scored by mAP at IoU 0.5. You read the XML, convert every box into YOLO's label format, build the folder and the data YAML that ultralytics expects, train YOLOv8n from its published COCO weights, read the training curves, score the validation split per class, look at the boxes it draws, and write a submission with one row per box.

**Your role:** Fill in the `# Your code here` cells in order. Run every cell, including the ones already written for you.

**Dataset:** Kaggle `andrewmvd/road-sign-detection` (about 230 MB), downloaded by `kagglehub` in the setup cell: `images/roadN.png` (mostly 300 x 400) and `annotations/roadN.xml` in PASCAL VOC format. You split 700 photos for training and 177 for validation. Use a GPU (Runtime, Change runtime type, T4): the 15 epochs take a few minutes there and much longer on a CPU.

---
# Setup

In [ ]:
# ultralytics is the YOLO library; Colab has everything else.
!pip install -q ultralytics kagglehub

In [ ]:
import os, shutil, random
import xml.etree.ElementTree as ET
from collections import Counter
import yaml
import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image
import torch
from ultralytics import YOLO
random.seed(0)
device = 0 if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)

In [ ]:
path = kagglehub.dataset_download("andrewmvd/road-sign-detection")
IMG, ANN = os.path.join(path, "images"), os.path.join(path, "annotations")
CLASSES = ["trafficlight", "stop", "speedlimit", "crosswalk"]       # YOLO numbers classes from 0 in this order
ids = sorted(int(f[4:-4]) for f in os.listdir(IMG))                 # road0.png -> 0
print(len(ids), "photos;", open(os.path.join(ANN, "road0.xml")).read()[:420])

---
# Part 1 -- Boxes from XML

The *A Box Is Four Numbers* slide. VOC XML stores the image `size` and, for every object, a `name` and a `bndbox` with `xmin, ymin, xmax, ymax`: the two-corner format.

## Task 1: Parse one XML file

Write `parse_xml(xml_path)` that returns `(width, height, objects)`: `width` and `height` as integers from the `size` element, and `objects` as a list with one tuple `(name, xmin, ymin, xmax, ymax)` per `object` element, the four corners as integers. Test it on `road0.xml`: a 267 x 400 photo with one `trafficlight` box `(98, 62, 208, 232)`.

Syntax hint (the shape of the call, not the answer):

```python
root = ET.parse(path).getroot()
value = int(root.find("size").find("width").text)
for obj in root.findall("object"):
    name = obj.find("name").text
```

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert (w0, h0) == (267, 400) and obj0 == [('trafficlight', 98, 62, 208, 232)], 'parse_xml: width and height from size; one (name, xmin, ymin, xmax, ymax) tuple per object'
print('Task 1 passed')

## Task 2: Count the classes and look at the boxes

Count how many boxes of each class the 877 files contain (a `Counter` over the names returned by `parse_xml` for every file) and print it. Then show 4 photos with their boxes drawn: `ax.add_patch(Rectangle((xmin, ymin), xmax - xmin, ymax - ymin, fill=False, color="orange", linewidth=2))` draws one box on an `imshow` axis, and `ax.text(xmin, ymin - 3, name, color="orange")` writes the class above it. Which class dominates?

Syntax hint (the shape of the call, not the answer):

```python
ax.imshow(image)
ax.add_patch(Rectangle((x, y), width, height, fill=False, color="orange", linewidth=2))
```

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert counts['speedlimit'] == 783 and sum(counts.values()) == 1244, 'counts: parse every XML file; 1,244 boxes in total, 783 of them speed limits'
print('Task 2 passed')

---
# Part 2 -- YOLO's format

The *In Code: YOLO's Files and the Three Calls* slide. YOLO wants one text file per image with one line per box, `class cx cy w h`, the centre and the size as fractions of the image. Getting this conversion right is most of the work in a detection task.

## Task 3: Corners to YOLO fractions

Write `to_yolo(x1, y1, x2, y2, w, h)` that returns `(cx, cy, bw, bh)`: the box centre divided by the image width and height, and the box width and height divided by the same. Test it on the `road0.xml` box: `(98, 62, 208, 232)` in a 267 x 400 photo gives `cx = 153 / 267 = 0.573`, `cy = 147 / 400 = 0.3675`, `bw = 110 / 267 = 0.412`, `bh = 170 / 400 = 0.425`, the numbers from the Olympiad Practice slide.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert all(abs(a - b) < 1e-3 for a, b in zip(line0, (0.5730, 0.3675, 0.4120, 0.4250))), 'to_yolo: centre and size divided by the image width (x) and height (y)'
print('Task 3 passed')

## Task 4: Build the dataset folder

Shuffle a copy of `ids` with `random.shuffle` (the seed is set in the setup cell) and take the first 700 as `train_ids`, the rest as `val_ids`. Create `signs/images/train`, `signs/images/val`, `signs/labels/train`, `signs/labels/val`. For every id in each split, copy `roadN.png` into the split's `images` folder and write `roadN.txt` into its `labels` folder with one line per box: the class index from `CLASSES`, then the four fractions from `to_yolo`, formatted with 6 decimals and separated by spaces. Print how many files each of the four folders holds, and the content of one label file.

Syntax hint (the shape of the call, not the answer):

```python
with open(path, "w") as f:
    f.write("\n".join(lines) + "\n")       # lines is a list of strings such as "2 0.5 0.5 0.2 0.3"
```

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
n = {s: (len(os.listdir(os.path.join(ROOT, 'images', s))), len(os.listdir(os.path.join(ROOT, 'labels', s)))) for s in ('train', 'val')}
assert n['train'] == (700, 700) and n['val'] == (177, 177), 'folders: 700 train and 177 val photos, one label file per photo'
assert open(os.path.join(ROOT, 'labels', split0, 'road0.txt')).read().startswith('0 0.573'), 'road0.txt: class 0 (trafficlight) then the four fractions'
print('Task 4 passed')

The data YAML tells ultralytics where the folders are and what the class numbers mean. Its `path` must be absolute: a relative one is resolved against the library's own datasets folder, not your notebook, and the training stops with `Dataset not found`.

In [ ]:
data = {"path": ROOT, "train": "images/train", "val": "images/val", "names": {i: c for i, c in enumerate(CLASSES)}}
YAML = os.path.abspath("signs.yaml")
with open(YAML, "w") as f:
    yaml.safe_dump(data, f, sort_keys=False)
print(open(YAML).read())

## Task 5: Draw the labels back

Before training, check the files the way the Olympiad Practice slide says: write `from_yolo(cx, cy, bw, bh, w, h)` that turns one label line back into integer corners `(x1, y1, x2, y2)`, then for the first 4 validation photos read `signs/labels/val/roadN.txt`, convert every line back to corners (the image size comes from `Image.open(...).size`) and draw them with `show_boxes` in purple. The boxes must sit on the signs exactly as the orange ones did in Task 2.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert from_yolo(*line0, w0, h0) == (98, 62, 208, 232), 'from_yolo: corners are centre minus and plus half the size, scaled back by width and height'
print('Task 5 passed')

---
# Part 3 -- Train YOLOv8n

The *From the 2016 Paper to Today's YOLO* slide: the smallest size, `n`, starts from its published COCO weights. The training loop, the loss and the augmentation live inside `train`; you choose the data, the epochs, the image size and the batch.

## Task 6: Load the pretrained model

Create `model = YOLO("yolov8n.pt")` (the weights download on first use). Print the number of parameters, `sum(p.numel() for p in model.model.parameters())`, and the number of classes it knows, `len(model.names)`. Compare the count with the table on the slide.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert 3_000_000 < n_params < 3_400_000 and len(model.names) == 80, 'YOLO("yolov8n.pt"): about 3.2 million parameters and 80 COCO classes'
print('Task 6 passed')

## Task 7: Train

Call `model.train(data=YAML, epochs=15, imgsz=416, batch=16, seed=0, device=device, project="runs_signs", name="yolov8n", exist_ok=True)` and keep the returned object as `results`. Watch the table it prints after every epoch: `box_loss` and `cls_loss` should fall and `mAP50` should climb past 0.7. Print `results.save_dir`: that folder holds `weights/best.pt` and `results.csv`.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert os.path.exists(os.path.join(SAVE, 'weights', 'best.pt')) and os.path.exists(os.path.join(SAVE, 'results.csv')), 'train: results.save_dir must contain weights/best.pt and results.csv'
print('Task 7 passed')

## Task 8: Read the curves

Read `results.csv` from `SAVE` into a DataFrame and strip the spaces from its column names (`df.columns = df.columns.str.strip()`). Plot two panels against `epoch`: the training losses `train/box_loss` and `train/cls_loss` on the left, `metrics/mAP50(B)` and `metrics/mAP50-95(B)` on the right, with legends and titles. Store the last `metrics/mAP50(B)` as `final_map50`. Which curve is still moving at epoch 15, and what would you try next?

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert final_map50 > 0.5, 'curves: mAP@0.5 should pass 0.5 well before epoch 15; if not, check the label files of Task 4 and the YAML'
print('Task 8 passed')

---
# Part 4 -- Score, look, submit

The *Average Precision and mAP* slide: mAP@0.5 is the mean over the four classes of the area under each class's precision-recall curve, boxes counting as hits at IoU 0.5.

## Task 9: Validation mAP, per class

Load the best checkpoint, `best = YOLO(os.path.join(SAVE, "weights", "best.pt"))`, and run `m = best.val(data=YAML, imgsz=416, device=device)`. Print `m.box.map50` and `m.box.map` (mAP@[.5:.95]), then one line per class with its AP@0.5: `m.box.ap_class_index` lists the class numbers and `m.box.ap50` their APs in the same order. Which class is hardest, and what in the counts of Task 2 explains it?

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert m.box.map50 > 0.5 and len(m.box.ap50) == 4, 'val: map50 above 0.5 and one AP per class'
print('Task 9 passed')

## Task 10: Draw the predictions

For the first 4 validation photos run `best.predict(img_path, conf=0.5, imgsz=416, device=device, verbose=False)[0]`; `r.plot()` returns the photo with the boxes drawn as a BGR array, so show `r.plot()[:, :, ::-1]`. Print, for each photo, `r.boxes.xyxy` rounded, `r.boxes.conf` and `r.boxes.cls`. Compare with the purple boxes of Task 5: where does it miss, and where is it unsure?

In [ ]:
# Your code here


## Task 11: Write the submission

Run `best.predict` on every validation photo with `conf=0.3` and collect one row per box: `image` (`roadN.png`), `label` (the class name from `r.boxes.cls`), `x1, y1, x2, y2` (from `r.boxes.xyxy`, rounded to one decimal) and `score` (`r.boxes.conf`, three decimals). Save the rows as `submission.csv` without the index, read it back, and print its shape and first 5 rows. Every label must be one of the four class names.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
check = pd.read_csv('submission.csv')
assert set(check.columns) == {'image', 'label', 'x1', 'y1', 'x2', 'y2', 'score'} and set(check['label']) <= set(CLASSES) and len(check) > 100, 'submission: one row per box with score >= 0.3, class names as labels'
print('Task 11 passed')

---
## Task 12: Reflect

YOLOv8n reached a useful mAP after 15 epochs on 700 photos. Which part of the model was already trained before you started, and which part learned your four classes? Task 9 showed one class behind the others. Name the two reasons the data suggests, and one move for each. A competition scored by mAP@0.5 rewards a box that overlaps the truth by half; mAP@[.5:.95] does not. Which of your predictions in Task 10 would lose most points under the stricter rule? In Task 4 you wrote the label files yourself. Name two mistakes in that step that would make training run without an error but give a low mAP.

*Your answers here*

---
## Conclusion

- **Boxes from files**: VOC XML parsed into corners and class names, drawn on the photos before any training.
- **YOLO's format**: corners to `class cx cy w h` fractions and back, one label file per photo, a data YAML with an absolute path, and the files drawn back to check them.
- **A pretrained detector trained with three calls**: `train` from the COCO weights, the curves in `results.csv`, `val` for mAP@0.5 per class.
- **Looking and handing in**: `predict` with a confidence threshold, the boxes drawn, and a submission with one row per box.

---

Made By **Sattam Altwaim**